In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

In [2]:
app = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")
prev = pd.read_csv("../data/previous_application.csv")

print(f"Application:          {app.shape}")
print(f"Bureau:               {bureau.shape}")
print(f"Previous application: {prev.shape}")

Application:          (307511, 122)
Bureau:               (1716428, 17)
Previous application: (1670214, 37)


In [3]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_count=('SK_ID_BUREAU', 'count'),
    bureau_active_count=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_closed_count=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    bureau_amt_credit_sum=('AMT_CREDIT_SUM', 'sum'),
    bureau_amt_credit_mean=('AMT_CREDIT_SUM', 'mean'),
    bureau_amt_credit_max=('AMT_CREDIT_SUM', 'max'),
    bureau_amt_debt_sum=('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_overdue_max=('AMT_CREDIT_SUM_OVERDUE', 'max'),
    bureau_days_credit_max=('DAYS_CREDIT', 'max'),
    bureau_days_credit_min=('DAYS_CREDIT', 'min'),
)
bureau_agg['bureau_debt_credit_ratio'] = (
    bureau_agg['bureau_amt_debt_sum'] / bureau_agg['bureau_amt_credit_sum'].replace(0, np.nan)
)
print(f"Bureau agg: {bureau_agg.shape}")

Bureau agg: (305811, 11)


In [4]:
prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_count=('SK_ID_PREV', 'count'),
    prev_approved_count=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    prev_refused_count=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_amt_application_mean=('AMT_APPLICATION', 'mean'),
    prev_amt_application_max=('AMT_APPLICATION', 'max'),
    prev_amt_credit_mean=('AMT_CREDIT', 'mean'),
    prev_amt_credit_max=('AMT_CREDIT', 'max'),
    prev_days_decision_max=('DAYS_DECISION', 'max'),
    prev_days_decision_min=('DAYS_DECISION', 'min'),
)

# Derived features
prev_agg['prev_refusal_rate'] = (
    prev_agg['prev_refused_count'] / prev_agg['prev_count']
)
prev_agg['prev_credit_vs_application'] = (
    prev_agg['prev_amt_credit_mean'] / prev_agg['prev_amt_application_mean'].replace(0, np.nan)
)

print(f"Previous app agg: {prev_agg.shape}")
prev_agg.head()

Previous app agg: (338857, 11)


,prev_count,prev_approved_count,prev_refused_count,prev_amt_application_mean,prev_amt_application_max,prev_amt_credit_mean,prev_amt_credit_max,prev_days_decision_max,prev_days_decision_min,prev_refusal_rate,prev_credit_vs_application
SK_ID_CURR,,,,,,,,,,,
100001,1,1,0,24835.50,24835.5,23787.00,23787.0,-1740,-1740,0.0,0.957782
100002,1,1,0,179055.00,179055.0,179055.00,179055.0,-606,-606,0.0,1.000000
100003,3,3,0,435436.50,900000.0,484191.00,1035882.0,-746,-2341,0.0,1.111967
100004,1,1,0,24282.00,24282.0,20106.00,20106.0,-815,-815,0.0,0.828021
100005,2,1,0,22308.75,44617.5,20076.75,40153.5,-315,-757,0.0,0.899950


In [5]:
df = app.merge(bureau_agg, on='SK_ID_CURR', how='left')
df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
print(f"Merged shape: {df.shape}")

Merged shape: (307511, 144)


In [6]:
feature_cols = [c for c in df.columns if c not in ['TARGET', 'SK_ID_CURR']]
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(include=['object']).columns.tolist()

print(f"Numeric: {len(numeric_features)}, Categorical: {len(categorical_features)}")

X = df[feature_cols]
y = df['TARGET']

Numeric: 126, Categorical: 16


C:\Users\frana\AppData\Local\Temp\ipykernel_14208\4011276221.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df[feature_cols].select_dtypes(include=['object']).columns.tolist()


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced")),
])

pipe.fit(X_train, y_train)

y_prob = pipe.predict_proba(X_test)[:, 1]
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUC-PR:  {average_precision_score(y_test, y_prob):.4f}")
print(f"Iterations used: {pipe.named_steps['model'].n_iter_}")

AUC-ROC: 0.7581
AUC-PR:  0.2402
Iterations used: [274]
